In [2]:
import os
import time
import joblib
import warnings
import numpy as np
import pandas as pd
from sklearn import metrics
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix  
from sklearn.metrics import classification_report 
warnings.filterwarnings(action = 'ignore')
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

In [3]:
df=pd.read_csv('covid19_1.csv')

In [4]:
df

,Unnamed: 0,SEX,BP_high,BP_low,RR,TEMP,SPO2,covid,Age
0,0,Female,120,80,20.000000,100.000000,85.000000,Yes,5
1,1,Female,110,70,20.000000,88.871658,97.000000,Yes,10
2,2,Male,120,80,20.733591,88.871658,94.184211,Yes,9
3,3,Male,110,80,18.000000,98.000000,98.000000,Yes,10
4,4,Male,130,86,18.000000,100.000000,96.000000,Yes,8
...,...,...,...,...,...,...,...,...,...
269,25,Male,124,83,24.000000,37.000000,94.000000,NO,16
270,26,Female,139,83,22.000000,37.000000,74.000000,NO,10
271,27,Female,94,56,18.000000,37.000000,94.000000,NO,7
272,28,Male,110,68,18.000000,40.000000,95.000000,NO,9


In [5]:
df.shape

(274, 9)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 274 entries, 0 to 273
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  274 non-null    int64  
 1   SEX         274 non-null    object 
 2   BP_high     274 non-null    int64  
 3   BP_low      274 non-null    int64  
 4   RR          274 non-null    float64
 5   TEMP        274 non-null    float64
 6   SPO2        274 non-null    float64
 7   covid       274 non-null    object 
 8   Age         274 non-null    int64  
dtypes: float64(3), int64(4), object(2)
memory usage: 19.4+ KB


In [7]:
df.columns

Index(['Unnamed: 0', 'SEX', 'BP_high', 'BP_low', 'RR', 'TEMP', 'SPO2', 'covid',
       'Age'],
      dtype='object')

In [8]:
df['SEX'].unique()

array(['Female', 'Male'], dtype=object)

In [9]:
df['SEX'].replace({'Female':1,'Male':0},inplace=True)

In [10]:
df['SEX'].unique()

array([1, 0])

In [11]:
df['covid'].unique()

array(['Yes', 'NO'], dtype=object)

In [12]:
df['covid'].replace({'Yes':1,'NO':0},inplace=True)

In [13]:
df['covid'].unique()

array([1, 0])

In [14]:
df['RR']=df['RR'].astype(int)

In [15]:
df['TEMP']=df['TEMP'].astype(int)

In [16]:
df['SPO2']=df['SPO2'].astype(int)

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 274 entries, 0 to 273
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Unnamed: 0  274 non-null    int64
 1   SEX         274 non-null    int64
 2   BP_high     274 non-null    int64
 3   BP_low      274 non-null    int64
 4   RR          274 non-null    int64
 5   TEMP        274 non-null    int64
 6   SPO2        274 non-null    int64
 7   covid       274 non-null    int64
 8   Age         274 non-null    int64
dtypes: int64(9)
memory usage: 19.4 KB


In [18]:
df.describe()

,Unnamed: 0,SEX,BP_high,BP_low,RR,TEMP,SPO2,covid,Age
count,274.000000,274.000000,274.000000,274.000000,274.000000,274.000000,274.000000,274.000000,274.000000
mean,109.784672,0.390511,119.390511,75.167883,20.693431,88.594891,94.178832,0.890511,9.861314
std,74.584473,0.488758,27.201811,16.453762,9.767676,18.729229,6.992994,0.312823,3.933853
min,0.000000,0.000000,0.000000,0.000000,12.000000,36.000000,40.000000,0.000000,1.000000
25%,38.250000,0.000000,110.000000,70.000000,17.000000,88.000000,94.000000,1.000000,7.000000
50%,106.500000,0.000000,120.000000,80.000000,18.000000,98.000000,96.000000,1.000000,10.000000
75%,174.750000,1.000000,130.000000,80.000000,20.000000,99.000000,98.000000,1.000000,13.000000
max,243.000000,1.000000,200.000000,100.000000,90.000000,102.000000,100.000000,1.000000,21.000000


In [19]:
X=df.drop('covid',axis=1)
y=df['covid']

In [20]:
X

,Unnamed: 0,SEX,BP_high,BP_low,RR,TEMP,SPO2,Age
0,0,1,120,80,20,100,85,5
1,1,1,110,70,20,88,97,10
2,2,0,120,80,20,88,94,9
3,3,0,110,80,18,98,98,10
4,4,0,130,86,18,100,96,8
...,...,...,...,...,...,...,...,...
269,25,0,124,83,24,37,94,16
270,26,1,139,83,22,37,74,10
271,27,1,94,56,18,37,94,7
272,28,0,110,68,18,40,95,9


In [21]:
y

0      1
1      1
2      1
3      1
4      1
      ..
269    0
270    0
271    0
272    0
273    0
Name: covid, Length: 274, dtype: int64

In [22]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X, y, test_size = 0.3, random_state = 0)

In [23]:
from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state = 2)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train.ravel())

In [24]:
knn = KNeighborsClassifier(n_neighbors=6)

In [25]:
knn.fit(X_train_res,y_train_res)

KNeighborsClassifier(n_neighbors=6)

In [26]:
y_pred = knn.predict(X_test)
print("Prediction : {}".format(y_pred))

Prediction : [1 0 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 0 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1
 0 1 1 1 1 0 1 1 0]


In [27]:
print("Covid Accuracy:",metrics.accuracy_score(y_test, y_pred)*100)

Covid Accuracy: 100.0


In [28]:
joblib.dump(knn,'covid_model.pkl')

['covid_model.pkl']

In [29]:
results = confusion_matrix(y_test, y_pred) 
print('Confusion Matrix :')
print(results) 
print('Accuracy Score :',accuracy_score(y_test, y_pred))
print('Report :')
print(classification_report(y_test, y_pred))

Confusion Matrix :
[[ 8  0]
 [ 0 75]]
Accuracy Score : 1.0
Report :
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         8
           1       1.00      1.00      1.00        75

    accuracy                           1.00        83
   macro avg       1.00      1.00      1.00        83
weighted avg       1.00      1.00      1.00        83



In [30]:
from sklearn import datasets
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
import xgboost
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

ModuleNotFoundError: No module named 'xgboost'

In [ ]:
dtc =  DecisionTreeClassifier()
rfc = RandomForestClassifier()
knn =  KNeighborsClassifier()
xgb = xgboost.XGBClassifier()

In [ ]:
clf = [dtc,rfc,knn,xgb]
for algo in clf:
    score = cross_val_score( algo,X_train_res,y_train_res,cv = 5,scoring = 'accuracy')
    print("The accuracy score of {} is:".format(algo),score.mean())

In [ ]:
dtc =  DecisionTreeClassifier()
rfc = RandomForestClassifier()
knn =  KNeighborsClassifier()
xgb = xgboost.XGBClassifier()
clf = [('dtc',dtc),('rfc',rfc),('knn',knn),('xgb',xgb)]

In [ ]:
clf

In [ ]:
for i in clf:
    print(i[0])
    i[1].fit(X_train_res,y_train_res)

In [ ]:
import joblib
for i in clf:
    filename = ''+i[0]+'.sav'
    joblib.dump(i[1], filename)

In [ ]:
df['SEX'].unique()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


# Build the ANN model
model = Sequential()
model.add(Dense(32, activation='relu', input_dim=X_train.shape[1]))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, batch_size=32, epochs=10, verbose=1)

# Evaluate the model on the test set
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Test loss: {loss:.4f}')
print(f'Test accuracy: {accuracy:.4f}')


In [ ]:
model.save('ann.cvf5')

In [ ]:
import os
import time
import joblib
import warnings
import pandas as pd
warnings.filterwarnings(action = 'ignore')
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import load_model

def covid_model(covid_value):    
    classifier = load_model('ann.cvf5')
    for i in range(len(covid_value)):
        covid_value[i]=float(i)
        
    new_pred = classifier.predict([covid_value])
    if new_pred==[0.]:
        result = 'notpresent'
        print(result)
    else:
        result = 'present'
        print(result)
    f = open("output.txt","w")
    f.write(result)
    f.close()
    print('\n')


if __name__ == '__main__':
    TestData="covid_test"
    while True:
        for(direcpath,direcnames,files) in os.walk(TestData):
            for file in files:
                if 'txt' in file:
                    time.sleep(1)
                    filename= TestData +'/'+ file
                    print('Filename:',filename)
                    with open('input.txt', 'r') as myfile:
                        data = myfile.read()
                        testdata = data.split("\n")
                        clf = covid_model(testdata)
                    os.remove(filename)



In [ ]:
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RepeatedKFold

In [ ]:
model = Lasso()

In [ ]:
from pandas import read_csv
from sklearn.linear_model import Lasso
model = Lasso(alpha=1.0)
model.fit(X_train,y_train)

In [ ]:
X_test

In [ ]:
p = model.predict([X_test[244]])
p=round(abs(p[0]))
print('Predicted: %.3f' % p)